# Train the static baselines (Logistic Regression, SVM)

Set `CONDITION` and run the whole notebook; it writes
`results/static_<condition>.json`.

In [ ]:
CONDITION = "raw" # "po" = participant-only clips, "raw" = full interview recordings

import numpy as np
import common
from common import (SEGMENT_LENGTHS, load_metadata, load_features, run_lr, run_svm, random_baseline, save_results)

print("Condition:", CONDITION)
print("Features :", common.FEATURE_PATHS[CONDITION])

Condition: raw
Features : ./androids_is09_02.npz


In [8]:
interview_df, label_of, gender_of, interview_folds = load_metadata()
data = load_features(CONDITION)
print(f"Speakers: {len(data)}")

Speakers: 116


## Logistic Regression

In [9]:
lr_results = {}

for seg_len in SEGMENT_LENGTHS:
    smv_folds, wa_folds, pooled = run_lr(data, label_of, interview_folds, seg_len)
    lr_results[str(seg_len)] = {"smv": smv_folds, "wa": wa_folds, "pooled": pooled}

    f1_smv = np.mean([f["f1"] for f in smv_folds]) * 100
    f1_wa = np.mean([f["f1"] for f in wa_folds]) * 100
    print(f"seg_len {seg_len:>4}: F1 SMV {f1_smv:.1f}  |  F1 WA {f1_wa:.1f}")

seg_len   32: F1 SMV 60.9  |  F1 WA 63.2
seg_len   64: F1 SMV 60.8  |  F1 WA 63.2
seg_len  128: F1 SMV 60.8  |  F1 WA 63.2
seg_len  256: F1 SMV 59.6  |  F1 WA 63.2
seg_len  512: F1 SMV 60.2  |  F1 WA 63.2
seg_len 1024: F1 SMV 59.1  |  F1 WA 63.2


## SVM on whole-recording averages

In [10]:
svm_folds, svm_pooled = run_svm(data, label_of, interview_folds)

f1 = np.array([f["f1"] for f in svm_folds])
acc = np.array([f["acc"] for f in svm_folds])
print(f"SVM: acc {acc.mean()*100:.1f} ± {acc.std(ddof=1)*100:.1f}   "
      f"F1 {f1.mean()*100:.1f} ± {f1.std(ddof=1)*100:.1f}")

SVM: acc 80.2 ± 4.9   F1 81.4 ± 6.7


## Random baseline

In [11]:
rand = random_baseline(label_of, n_trials=1000, seed=0)
for key, val in rand.items():
    print(f"  {key:10s}: {val*100:.1f}")

  acc       : 50.6
  precision : 55.1
  recall    : 55.3
  f1        : 55.0


In [12]:
save_results({
    "condition": CONDITION,
    "segment_lengths": SEGMENT_LENGTHS,
    "lr": lr_results,
    "svm": {"folds": svm_folds, "pooled": svm_pooled},
    "random_baseline": rand,
}, f"static_{CONDITION}")

Saved results\static_raw.json


'results\\static_raw.json'